# 04 · Phase A pilot sampling (pilot/v1/mc3d-100)

Stratified random sampling of 40 small + 40 medium + 20 large MC3D structures (seed=42) into the `pilot/v1/mc3d-100` Group, plus the per-structure spin_type decision applied to the pilot set.

*Consolidated structure-sample.ipynb (2026-06-16 notebook cleanup).*

In [2]:


from aiida import load_profile, orm
from aiida.engine import submit
from aiida.orm import (
    Code, Group, InstalledCode, QueryBuilder, StructureData,
    WorkChainNode, load_code, load_group, load_node,
)
from aiida_quantumespresso.common.types import ElectronicType, SpinType
from aiida_quantumespresso.workflows.pw.base import PwBaseWorkChain

load_profile();


In [3]:
# Stratified random sampling: 40 small + 40 medium + 20 large from MC3D PBEsol v2.
# Seed=42 for reproducibility. Tags structures into pilot/v1/mc3d-100 Group.

import random
from aiida.orm import Group, QueryBuilder, StructureData, load_group
from collections import Counter

SEED = 42
PILOT_LABEL = 'pilot/v1/mc3d-100'

mc3d = load_group('mc3d-pbesol-v2-structures')

# Pull (uuid, n_atoms) for ALL structures — fast metadata-only query
qb = QueryBuilder()
qb.append(Group, filters={'label': mc3d.label}, tag='g')
qb.append(StructureData, with_group='g', project=['uuid', 'attributes.sites'])
rows = qb.all()

# Bin by n_atoms
small, medium, large = [], [], []
for uuid, sites in rows:
    n = len(sites)
    if   n <=  8: small.append(uuid)
    elif n <= 20: medium.append(uuid)
    else:         large.append(uuid)
print(f'Population:  small={len(small)}  medium={len(medium)}  large={len(large)}')

rng = random.Random(SEED)
picked = (
    rng.sample(small,  min(40, len(small)))  +
    rng.sample(medium, min(40, len(medium))) +
    rng.sample(large,  min(20, len(large)))
)
print(f'Picked: {len(picked)} structures')

# Resolve UUIDs back to nodes
qb = QueryBuilder()
qb.append(StructureData, filters={'uuid': {'in': picked}})
nodes = qb.all(flat=True)

# Tag into pilot Group
pilot, created = Group.collection.get_or_create(label=PILOT_LABEL)
if created:
    pilot.description = f'Phase A pilot: 40 small + 40 medium + 20 large from MC3D, seed={SEED}'
pilot.add_nodes(nodes)
print(f"Group '{PILOT_LABEL}' now contains {pilot.count()} structures")

# Sanity: element distribution + n_atoms summary
n_atoms_picked = [len(n.sites) for n in nodes]
element_counter = Counter()
for n in nodes:
    element_counter.update(n.get_symbols_set())

print(f'\nn_atoms: min={min(n_atoms_picked)}  median={sorted(n_atoms_picked)[len(n_atoms_picked)//2]}  max={max(n_atoms_picked)}')
print(f'Element coverage: {len(element_counter)} unique elements')
print(f'Top 10 elements: {element_counter.most_common(10)}')


Population:  small=8345  medium=10717  large=14080
Picked: 100 structures
Group 'pilot/v1/mc3d-100' now contains 100 structures

n_atoms: min=2  median=12  max=68
Element coverage: 58 unique elements
Top 10 elements: [('O', 30), ('B', 11), ('Zr', 10), ('N', 9), ('Si', 9), ('Al', 9), ('Ca', 8), ('Ba', 8), ('Li', 8), ('Nb', 8)]


In [4]:
# Phase A pilot: 40 small + 40 medium + 20 large, stratified by n_atoms.
# Tag into pilot/v1/mc3d-100 Group, seed=42 reproducible.

import random
from collections import Counter
from aiida import load_profile
from aiida.orm import Group, QueryBuilder, StructureData, load_group

load_profile();

SEED = 42
PILOT_LABEL = 'pilot/v1/mc3d-100'
QUOTA = {'small': 40, 'medium': 40, 'large': 20}

mc3d = load_group('mc3d-pbesol-v2-structures')

# Pull (uuid, n_atoms) for ALL structures — metadata-only, fast
qb = QueryBuilder()
qb.append(Group, filters={'label': mc3d.label}, tag='g')
qb.append(StructureData, with_group='g', project=['uuid', 'attributes.sites'])
rows = qb.all()

bins = {'small': [], 'medium': [], 'large': []}
for uuid, sites in rows:
    n = len(sites)
    if   n <=  8: bins['small'].append(uuid)
    elif n <= 20: bins['medium'].append(uuid)
    else:         bins['large'].append(uuid)
print(f'Population:  small={len(bins["small"])}  medium={len(bins["medium"])}  large={len(bins["large"])}')

rng = random.Random(SEED)
picked_uuids = []
for tier, q in QUOTA.items():
    pool = bins[tier]
    if len(pool) < q:
        print(f'  WARN: {tier} pool only has {len(pool)}, requested {q}')
        q = len(pool)
    picked_uuids += rng.sample(pool, q)
print(f'Picked: {len(picked_uuids)} structures')

# Resolve UUIDs → StructureData
qb = QueryBuilder()
qb.append(StructureData, filters={'uuid': {'in': picked_uuids}})
nodes = qb.all(flat=True)

# Tag into pilot group (idempotent — re-running with same SEED gives same set)
pilot, created = Group.collection.get_or_create(label=PILOT_LABEL)
if created:
    pilot.description = (
        f'Phase A pilot: 40 small + 40 medium + 20 large from MC3D, '
        f'stratified by n_atoms, seed={SEED}'
    )
pilot.add_nodes(nodes)
print(f"Group '{PILOT_LABEL}' contains {pilot.count()} structures (PK={pilot.pk})")

# Sanity stats
n_atoms_list = [len(n.sites) for n in nodes]
elem_counter = Counter()
for n in nodes:
    elem_counter.update(n.get_symbols_set())
print(f'\nn_atoms: min={min(n_atoms_list)}, median={sorted(n_atoms_list)[50]}, max={max(n_atoms_list)}')
print(f'Element coverage: {len(elem_counter)} unique elements across pilot')
print(f'Top 10 elements: {elem_counter.most_common(10)}')


Population:  small=8345  medium=10717  large=14080
Picked: 100 structures
Group 'pilot/v1/mc3d-100' contains 100 structures (PK=33)

n_atoms: min=2, median=12, max=68
Element coverage: 58 unique elements across pilot
Top 10 elements: [('O', 30), ('B', 11), ('Zr', 10), ('N', 9), ('Si', 9), ('Al', 9), ('Ca', 8), ('Ba', 8), ('Li', 8), ('Nb', 8)]


In [5]:
# Per-pilot-structure spin_type decision.
# Rule (Phase A first cut, no archive yet):
#   odd-electron   → SpinType.COLLINEAR  (hard physics constraint)
#   even-electron  → SpinType.NONE       (placeholder; archive import later
#                                         may upgrade some to COLLINEAR if magnetic)

from aiida_quantumespresso.common.types import SpinType

pseudo_group = load_group('PseudoDojo/0.4/PBEsol/SR/standard/upf')
pilot = load_group(PILOT_LABEL)


def n_valence_electrons(structure, pseudo_family) -> int:
    pseudos = pseudo_family.get_pseudos(structure=structure)
    return int(sum(pseudos[s.kind_name].z_valence for s in structure.sites))


def derive_spin_type(structure, pseudo_family):
    n_e = n_valence_electrons(structure, pseudo_family)
    is_odd = (n_e % 2) == 1
    return SpinType.COLLINEAR if is_odd else SpinType.NONE, n_e, is_odd


# Walk pilot group
pilot_meta = []
for s in pilot.nodes:
    spin, n_e, is_odd = derive_spin_type(s, pseudo_group)
    pilot_meta.append({
        'uuid':        s.uuid[:8],
        'pk':          s.pk,
        'formula':     s.get_formula(),
        'n_atoms':     len(s.sites),
        'n_electrons': n_e,
        'is_odd':      is_odd,
        'spin_type':   spin.value,           # 'none' or 'collinear'
        'elements':    sorted(s.get_symbols_set()),
    })

# Summary
n_odd = sum(m['is_odd'] for m in pilot_meta)
print(f'Pilot pool: {len(pilot_meta)} structures')
print(f'  odd-electron  → SpinType.COLLINEAR : {n_odd:3} ({100*n_odd/len(pilot_meta):.1f}%)')
print(f'  even-electron → SpinType.NONE      : {len(pilot_meta)-n_odd:3}')


Pilot pool: 100 structures
  odd-electron  → SpinType.COLLINEAR :  17 (17.0%)
  even-electron → SpinType.NONE      :  83


In [6]:
import pandas as pd

df = pd.DataFrame(pilot_meta)
df_sorted = df.sort_values(['n_atoms', 'formula']).reset_index(drop=True)

print(f'Pilot dataset: {len(df_sorted)} structures\n')

# Show by tier
for tier, lo, hi in [('small', 1, 8), ('medium', 9, 20), ('large', 21, 999)]:
    sub = df_sorted[(df_sorted.n_atoms >= lo) & (df_sorted.n_atoms <= hi)]
    n_odd_tier = sub.is_odd.sum()
    print(f'━━━ {tier:6s} ({lo}–{hi if hi<999 else "∞"} atoms): {len(sub)} structures, '
          f'{n_odd_tier} odd ━━━')
    print(sub[['pk', 'formula', 'n_atoms', 'n_electrons', 'is_odd', 'spin_type']].to_string(index=False))
    print()

# Save to CSV for record
out_csv = 'data/pilot/v1/mc3d-100-meta.csv'
import os
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df_sorted.drop(columns=['uuid']).to_csv(out_csv, index=False)
print(f'Saved: {out_csv}')


Pilot dataset: 100 structures

━━━ small  (1–8 atoms): 40 structures, 11 odd ━━━
   pk   formula  n_atoms  n_electrons  is_odd spin_type
 4339       AsY        2           26   False      none
23885       BZr        2           15    True collinear
  890      CdTe        2           36   False      none
20557       FeS        2           22   False      none
27007       NTc        2           20   False      none
 2370       Nb2        2           26   False      none
 4981       SbY        2           26   False      none
 1418      B2Be        3           10   False      none
13430    BaLiSi        3           17    True collinear
19462      CAgN        3           28   False      none
26212    CoCrTe        3           47    True collinear
15954    CuMgSn        3           43    True collinear
30863     TaTe2        3           45    True collinear
16349     Al3Zr        4           21    True collinear
14092     B2Pt2        4           42   False      none
11043     CrNb3        